# Fase 2b — Chains, prompting e confiabilidade

Rubrica: 2.3 (chains sequenciais, paralelas e router), 2.5 (PTCF + few-shot + chain-of-thought), 2.6 (parser com tratamento de erro, memória, ferramenta FAISS, callbacks).

## 0. Setup

In [1]:
import os, sys, time
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")

from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
    RunnableBranch,
)

from src.tools import search_video, search_wikipedia

llm = ChatGroq(model=os.getenv("GROQ_MODEL"), temperature=0, max_retries=6)
print(llm.invoke("ping").content[:50])

pong


In [2]:
class MestreResponse(BaseModel):
    level: Literal["BEGINNER", "ADVANCED"] = Field(
        description="Student level, from the identification rules"
    )
    context: str = Field(
        description="One sentence on what is happening and the goal; for a concept question, the short answer; for an off-topic question, the polite decline"
    )
    steps: list[str] | None = Field(
        default=None,
        max_length=5,
        description="Steps in order, at most 5; empty for concept or off-topic questions",
    )
    common_mistake: str | None = Field(
        default=None, description="BEGINNER only: one common mistake"
    )
    details: list[str] | None = Field(
        default=None,
        description="ADVANCED only: grips, angles, timing and the opponent's reactions",
    )
    variations: list[str] | None = Field(
        default=None,
        description="ADVANCED only: variations and counters you are confident about",
    )
    safety: str | None = Field(
        default=None, description="Safety reminder when submissions are involved"
    )
    clarifying_question: str | None = Field(
        default=None,
        description="Rule 4: the ONE clarifying question; when filled, leave steps, details and variations empty",
    )


parser = PydanticOutputParser(pydantic_object=MestreResponse)

chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """# PERSONA
You are Mestre, a Brazilian Jiu-Jitsu black belt instructor with many years on the mats and solid MMA knowledge. You are patient, helpful and direct. You use mat terminology (closed guard, sweep, guard pass, submission) and explain every term when the student is a beginner.

# TASK
Answer questions about Jiu-Jitsu and MMA techniques, defenses, escapes, positions and concepts, adapting the answer to the student's level.

# CONTEXT

## Sources
You have no tools or knowledge base yet. Use only well-established, widely taught Jiu-Jitsu knowledge. A video link is shown to the student separately, after your answer, so never write links yourself.

## Anti-hallucination rules
1. Never invent a technique, position name, step, statistic, athlete or link.
2. Never write, build or guess a URL.
3. If you are not confident about a detail, say "I'm not sure about this detail." and suggest checking it with the student's academy instructor. Never fill the gap with assumptions.
4. If an essential detail is missing (position, gi or no-gi, top or bottom) AND it changes the answer, ask ONE short clarifying question instead of answering.
5. If the question is not about Jiu-Jitsu, MMA or training, politely decline and steer back to the topic.

## Safety
When discussing submissions, remind the student to tap early, release at the first tap and train under supervision. Never give medical advice: for pain or injury, recommend a health professional.

## Student level identification
- BEGINNER: generic terms ("arm lock thing", "that move"), fundamentals, white belt or just started.
- ADVANCED: specific terminology (grips, variation names, kimura trap, berimbolo), describes the opponent's reactions, mentions competition rules, blue belt or higher.
If the student states their belt, the belt decides. When in doubt, treat them as a BEGINNER.

## Language
Always reply in the student's language.

# FORMAT
Reply ONLY with a JSON object that follows the schema below. Write every text value in the student's language.
- BEGINNER: fill context, steps (plain language), common_mistake and safety.
- ADVANCED: fill context, steps, details and variations; no explanation of basic terms.
- Leave a field empty when it does not apply.

{format_instructions}""",
        ),
        ("human", "{question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

search_prompt = PromptTemplate.from_template(
    "Turn this Jiu-Jitsu question into a short YouTube search query in English: "
    "technique + position + bjj, at most 6 words. Reply with the query only.\n\n"
    "Question: {question}"
)
search_chain = search_prompt | llm | StrOutputParser()

chain = chat_prompt | llm | parser

## 1. Chain sequencial (2.3)

Classificador de intenção (`Intent`) → a saída alimenta o próximo passo. `reasoning` vem antes de `intent` no schema (CoT explícito, usado no passo 4). v1 = zero-shot, baseline.

In [3]:
class Intent(BaseModel):
    reasoning: str = Field(
        description="One or two short sentences explaining the classification"
    )
    intent: Literal[
        "technique", "video", "history", "stats", "clarify", "off_topic"
    ] = Field(
        description="technique: how to do/defend/escape; video: wants to watch; history: athletes, academies, history; "
        "stats: competition numbers (ADCC); clarify: essential detail missing; off_topic: not BJJ/MMA"
    )
    search_term: str | None = Field(
        default=None,
        description="Short English keywords for tools, e.g. 'kimura closed guard bjj'",
    )


intent_parser = PydanticOutputParser(pydantic_object=Intent)

intent_prompt_v1 = ChatPromptTemplate.from_messages(
    [
        ("system", "Classify the Jiu-Jitsu question.\n{format_instructions}"),
        ("human", "{question}"),
    ]
).partial(format_instructions=intent_parser.get_format_instructions())

intent_chain_v1 = intent_prompt_v1 | llm | intent_parser

print(
    intent_chain_v1.invoke(
        {"question": "Quero ver um vídeo de kimura da guarda fechada"}
    )
)

reasoning='The user requests to watch a video of a kimura from closed guard.' intent='video' search_term='kimura closed guard'


In [4]:
def act_on_intent(x: dict) -> str:
    info = x["info"]
    if info.intent == "video":
        return search_video.invoke(info.search_term or x["question"])
    return f"no video needed ({info.intent})"


seq_chain = RunnablePassthrough.assign(info=intent_chain_v1) | RunnableLambda(
    act_on_intent
)

for q in ["Quero ver um vídeo de fuga da montada", "Quem foi Helio Gracie?"]:
    print(q, "->", seq_chain.invoke({"question": q}))
    time.sleep(5)

Quero ver um vídeo de fuga da montada -> The First 3 Mount Escapes You Need To Know in BJJ — https://www.youtube.com/watch?v=SYel-mVSMAI
Quem foi Helio Gracie? -> no video needed (history)


## 2. Chain paralela (2.3)

`RunnableParallel`: classificação e resposta do Mestre não dependem uma da outra → rodam ao mesmo tempo.

In [5]:
parallel_chain = RunnableParallel(
    info=intent_chain_v1,
    answer=chat_prompt | llm | parser,
)


def run_sequential(inputs: dict) -> dict:
    return {
        "info": intent_chain_v1.invoke(inputs),
        "answer": (chat_prompt | llm | parser).invoke(inputs),
    }


question = {"question": "Sou faixa branca. Como escapo da montada?"}

for name, fn in [("sequencial", run_sequential), ("paralelo", parallel_chain.invoke)]:
    time.sleep(15)  # deixa o limite de 8000 tokens/min do Groq recuperar
    start = time.perf_counter()
    result = fn(question)
    print(
        f"{name:10} {time.perf_counter() - start:.1f}s | intent={result['info'].intent} | level={result['answer'].level}"
    )

sequencial 2.6s | intent=technique | level=BEGINNER
paralelo   1.8s | intent=technique | level=BEGINNER


In [6]:
print(parallel_chain.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	Parallel\3cinfo\2canswer\3eInput([Parallel<info,answer>Input]):::first
	Parallel\3cinfo\2canswer\3eOutput([Parallel<info,answer>Output]):::last
	ChatPromptTemplate_1(ChatPromptTemplate)
	ChatGroq_1(ChatGroq)
	PydanticOutputParser_1(PydanticOutputParser)
	ChatPromptTemplate_2(ChatPromptTemplate)
	ChatGroq_2(ChatGroq)
	PydanticOutputParser_2(PydanticOutputParser)
	ChatPromptTemplate_1 --> ChatGroq_1;
	ChatGroq_1 --> PydanticOutputParser_1;
	Parallel\3cinfo\2canswer\3eInput --> ChatPromptTemplate_1;
	PydanticOutputParser_1 --> Parallel\3cinfo\2canswer\3eOutput;
	ChatPromptTemplate_2 --> ChatGroq_2;
	ChatGroq_2 --> PydanticOutputParser_2;
	Parallel\3cinfo\2canswer\3eInput --> ChatPromptTemplate_2;
	PydanticOutputParser_2 --> Parallel\3cinfo\2canswer\3eOutput;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. Router (2.3)

### 3.1 LCEL — `RunnableBranch`

O `Intent` decide o caminho; só o ramo escolhido roda.

In [7]:
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Mestre, a BJJ black belt instructor. Answer using ONLY the evidence below. "
               "If the answer is not in the evidence, say you are not sure. Reply in the student's language, in at most 4 sentences."),
    ("human", "Evidence:\n{evidence}\n\nQuestion: {question}"),
])

def fetch_wikipedia(x: dict) -> str:
    return search_wikipedia.invoke(x["info"].search_term or x["question"])

def fetch_video(x: dict) -> str:
    return search_video.invoke(x["info"].search_term or x["question"])

def stats_placeholder(x: dict) -> str:
    return "ADCC stats: coming in phase 4 (query_adcc)."

def decline_or_clarify(x: dict) -> str:
    if x["info"].intent == "clarify":
        return "Preciso de um detalhe: em qual posição você está (por cima/por baixo, guarda, montada)?"
    return "Só respondo sobre Jiu-Jitsu e MMA. Manda uma dúvida de técnica, posição ou história."

technique_chain = chat_prompt | llm | parser
history_chain = RunnablePassthrough.assign(evidence=fetch_wikipedia) | history_prompt | llm | StrOutputParser()
video_chain = RunnableLambda(fetch_video)
stats_chain = RunnableLambda(stats_placeholder)
fallback_chain = RunnableLambda(decline_or_clarify)


In [8]:
def route_is(name: str):
    return lambda x: x["info"].intent == name

router = RunnablePassthrough.assign(info=intent_chain_v1) | RunnableBranch(
    (route_is("technique"), technique_chain),
    (route_is("history"), history_chain),
    (route_is("video"), video_chain),
    (route_is("stats"), stats_chain),
    fallback_chain,  # default: clarify / off_topic
)

router_questions = [
    "Sou faixa branca. Como faço a raspagem de tesoura?",  # technique
    "Quem foi Helio Gracie?",                               # history
    "Me mostra um vídeo de triângulo da guarda fechada",    # video
    "Quantas finalizações por armlock teve no ADCC 2022?",  # stats
    "Qual a melhor receita de bolo de cenoura?",            # off_topic
]

for q in router_questions:
    out = router.invoke({"question": q})
    print(f">>> {q}\n    {type(out).__name__}: {str(out)[:250]}\n")
    time.sleep(8)


>>> Sou faixa branca. Como faço a raspagem de tesoura?
    MestreResponse: level='BEGINNER' context='Você está na guarda fechada e quer usar a raspagem de tesoura para levar o oponente ao topo.' steps=['Quebre a postura do oponente puxando a cabeça e controlando o braço dele.', 'Abra a guarda e coloque um dos seus pés na co

>>> Quem foi Helio Gracie?
    TextAccessor: Não tenho certeza quem foi Helio Gracie.

>>> Me mostra um vídeo de triângulo da guarda fechada
    str: Triangle from the Closed Guard? — https://www.youtube.com/watch?v=RVUL6KNBNG0

>>> Quantas finalizações por armlock teve no ADCC 2022?
    str: ADCC stats: coming in phase 4 (query_adcc).

>>> Qual a melhor receita de bolo de cenoura?
    str: Só respondo sobre Jiu-Jitsu e MMA. Manda uma dúvida de técnica, posição ou história.



In [9]:
# debug do ramo history: o que o classificador gerou e qual evidência chegou ao prompt
q = {"question": "Quem foi Helio Gracie?"}
for _ in range(2):
    info = intent_chain_v1.invoke(q)
    print("info:", info)
    print("evidence:", fetch_wikipedia({**q, "info": info})[:150])
    print("-" * 60)
    time.sleep(5)

info: reasoning='The question asks for information about a person important to BJJ history.' intent='history' search_term=None
evidence: Leandro Lo: Leandro Pereira do Nascimento Lo, better known as Leandro Lo, was a Brazilian jiu-jitsu black belt competitor. A record holder with eight 
------------------------------------------------------------
info: reasoning='The question asks for information about a person, which falls under historical background.' intent='history' search_term='Helio Gracie'
evidence: Hélio Gracie: Hélio Gracie was a Brazilian martial artist who together with his brothers Oswaldo, Gastao Jr, George and Carlos Gracie founded and deve
------------------------------------------------------------


**Diagnóstico do ramo history.** Com a mesma pergunta, o v1 às vezes devolve `search_term='Helio Gracie'` (evidência certa → resposta certa) e às vezes `search_term=None`. No `None`, o fallback manda a pergunta em português (`"Quem foi Helio Gracie?"`) para a Wikipedia em inglês, que devolve **Leandro Lo**. O prompt de história respondeu "não tenho certeza" em vez de inventar: a regra anti-alucinação funcionou, e o defeito está no classificador.

Correção no passo 4: o few-shot sempre preenche `search_term` nas intenções que usam ferramenta.

### 3.2 Router legado — `MultiPromptChain`

In [10]:
from langchain_classic.chains.router import MultiPromptChain

prompt_infos = [
    {
        "name": "technique",
        "description": "How to execute, defend or escape a BJJ technique or position",
        "prompt_template": "You are Mestre, a BJJ black belt instructor. Explain step by step, in the student's language.\n\nQuestion: {input}",
    },
    {
        "name": "history",
        "description": "BJJ history, athletes, academies and competitions",
        "prompt_template": "You are Mestre, a BJJ historian. Answer briefly, in the student's language. If unsure, say so.\n\nQuestion: {input}",
    },
]

legacy_router = MultiPromptChain.from_prompts(llm, prompt_infos, verbose=True)

for q in ["Quem foi Helio Gracie?", "Como faço a kimura da guarda fechada?"]:
    out = legacy_router.invoke({"input": q})
    print(f">>> {q}\n    {out['text'][:250]}\n")
    time.sleep(8)


D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langcha



> Entering new MultiPromptChain chain...
history: {'input': 'Quem foi Helio Gracie?'}
> Finished chain.
>>> Quem foi Helio Gracie?
    Helio Gracie (1913‑2009) foi um dos fundadores do Jiu‑Jitsu Brasileiro (BJJ). Ele adaptou o jiu‑jitsu tradicional japonês, criando técnicas que favoreciam pessoas menores e mais fracas, usando alavancas, posicionamento e finalizações no chão. Junto c



> Entering new MultiPromptChain chain...
technique: {'input': 'Como faço a kimura da guarda fechada?'}
> Finished chain.
>>> Como faço a kimura da guarda fechada?
    **Kimura da Guarda Fechada – Passo a Passo para o Aluno**

> *“A kimura é um dos ataques mais versáteis do Jiu‑Jitsu. Quando você a aplica da guarda fechada, tem a vantagem de estar protegido e ainda pode finalizar ou abrir a guarda para avançar.”*  



**LCEL vs legado:**

| | `RunnableBranch` | `MultiPromptChain` |
| --- | --- | --- |
| Decisão | Pydantic tipado (`Intent`), condição em Python | LLM devolve JSON em texto, `RouterOutputParser` |
| Destinos | qualquer Runnable (tool, chain, função) | só prompt → LLM |
| Status | API atual | deprecated (`langchain_classic`; cria `ConversationBufferMemory` por dentro → warnings) |
| Evidência | Wikipedia no ramo history | nenhuma: responde de memória, sem fonte verificável |
| Saída | `MestreResponse` validado no ramo technique | markdown livre (kimura veio como "tutorial de videogame", sem `safety`) |

## 4. Prompt PTCF + few-shot + chain-of-thought (2.5)

In [11]:
PTCF_INTENT = """# PERSONA
You are the intake assistant of Mestre, a Brazilian Jiu-Jitsu instructor. You route each student question to the right tool.

# TASK
Classify the question into ONE intent and extract the search term the tool will use.

# CONTEXT
Intents:
- technique: how to execute, defend, escape or finish a technique or position.
- video: the student wants to WATCH something (video, show me, mostra).
- history: athletes, academies, origins, who created something.
- stats: numbers from competitions (how common, how many, ADCC results).
- clarify: the answer depends on a detail the student did not give (position, top or bottom). Use it only when you really cannot answer.
- off_topic: not about BJJ/MMA techniques or history (cooking, diet, weight cutting, medical questions).

search_term rules:
- ALWAYS fill it for technique, video, history and stats. Leave it null only for clarify and off_topic.
- Always in English, even when the question is in Portuguese.
- technique / video / stats: technique + position + "bjj" (e.g. "triangle closed guard bjj").
- history: the person or subject only (e.g. "Rickson Gracie"), it goes to Wikipedia.

Think first: write the reasoning (1-2 short sentences), then decide the intent.

# FORMAT
{format_instructions}"""

examples = [
    {"question": "Me mostra como faz a kimura da guarda", "answer": '{"reasoning": "The student wants to SEE the kimura, so a video.", "intent": "video", "search_term": "kimura closed guard bjj"}'},
    {"question": "Como saio daí?", "answer": '{"reasoning": "No position is given and the escape depends on it.", "intent": "clarify", "search_term": null}'},
    {"question": "Who founded Gracie Barra?", "answer": '{"reasoning": "Asks about the origin of an academy.", "intent": "history", "search_term": "Gracie Barra"}'},
    {"question": "Faixa branca, como defendo o armlock da guarda fechada?", "answer": '{"reasoning": "Asks how to defend a submission from a known position.", "intent": "technique", "search_term": "armlock defense closed guard bjj"}'},
    {"question": "Heel hook finaliza muito no ADCC?", "answer": '{"reasoning": "Asks how common a submission is in competition, a number.", "intent": "stats", "search_term": "heel hook ADCC"}'},
    {"question": "Qual dieta pra secar pro campeonato?", "answer": '{"reasoning": "Diet and weight cut are outside techniques and history.", "intent": "off_topic", "search_term": null}'},
]

example_prompt = ChatPromptTemplate.from_messages([("human", "{question}"), ("ai", "{answer}")])
few_shot = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_prompt)

intent_prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", PTCF_INTENT),
    few_shot,
    ("human", "{question}"),
]).partial(format_instructions=intent_parser.get_format_instructions())

intent_chain = intent_prompt_v2 | llm | intent_parser


In [12]:
TOOL_INTENTS = {"technique", "video", "history", "stats"}

test_cases = [
    ("Me mostra o armlock", "video"),
    ("Como saio daqui? Ele tá me amassando", "clarify"),
    ("O armlock é comum no ADCC?", "stats"),
    ("Quem criou o berimbolo?", "history"),
    ("Quem foi Rickson Gracie?", "history"),
    ("Como perder 5kg pra competir no sábado?", "off_topic"),
    ("Tem algum vídeo explicando a passagem da De La Riva?", "video"),
    ("Faixa azul aqui. Como finalizo o triângulo quando o cara posturar?", "technique"),
]

def evaluate(chain, name):
    hits, terms_ok = 0, 0
    for question, expected in test_cases:
        info = chain.invoke({"question": question})
        hit = info.intent == expected
        term_ok = (info.search_term is not None) == (expected in TOOL_INTENTS)
        hits += hit
        terms_ok += term_ok
        print(f"{'OK ' if hit else 'ERR'} {question[:45]:45} -> {info.intent:9} | term={info.search_term} | {info.reasoning[:70]}")
        time.sleep(4)
    print(f"\n{name}: intent {hits}/{len(test_cases)} | search_term coerente {terms_ok}/{len(test_cases)}\n" + "=" * 80)

evaluate(intent_chain_v1, "v1 zero-shot")
time.sleep(20)
evaluate(intent_chain, "v2 PTCF + few-shot + CoT")


OK  Me mostra o armlock                           -> video     | term=arm lock BJJ | The user asks to be shown the armlock, which is a request for a demons
ERR Como saio daqui? Ele tá me amassando          -> technique | term=escape from mount | User asks how to escape a position where the opponent is crushing them
OK  O armlock é comum no ADCC?                    -> stats     | term=ADCC armlock frequency | The question asks about how common armlocks are in ADCC, which relates
OK  Quem criou o berimbolo?                       -> history   | term=who created the berimbolo origin | The question asks about the origin of the berimbolo, which is a histor
OK  Quem foi Rickson Gracie?                      -> history   | term=Rickson Gracie biography | The question asks for biographical information about Rickson Gracie, w
ERR Como perder 5kg pra competir no sábado?       -> technique | term=weight cut 5kg competition BJJ | The user asks how to lose weight quickly for a competition, which is a

In [13]:
router_v2 = RunnablePassthrough.assign(info=intent_chain) | router.steps[-1]  # mesmo RunnableBranch, classificador novo

for q in ["Quem foi Helio Gracie?", "Quem criou o berimbolo?"]:
    print(f">>> {q}\n    {router_v2.invoke({'question': q})}\n")
    time.sleep(8)


>>> Quem foi Helio Gracie?
    Hélio Gracie foi um artista marcial brasileiro que, junto com seus irmãos Oswaldo, Gastão Jr., George e Carlos Gracie, fundou e desenvolveu o sistema de defesa pessoal conhecido como Gracie Jiu‑Jitsu, também chamado de Brazilian Jiu‑Jitsu (BJJ).

>>> Quem criou o berimbolo?
    Não tenho certeza.



**Resultados (v1 vs v2, 8 casos ambíguos fora dos exemplos do few-shot):**

| Classificador | Intenção certa | `search_term` coerente |
| --- | --- | --- |
| v1 zero-shot | 6/8 | 6/8 |
| v2 PTCF + few-shot + CoT | 8/8 | 8/8 |

- v1 errou os 2 casos realmente ambíguos: "Como saio daqui? Ele tá me amassando" (sem posição → deveria ser `clarify`) e "Como perder 5kg pra competir?" (dieta → `off_topic`). Nos dois respondeu `technique`.
- v2 acertou porque o PTCF define cada intenção e diz quando usar `clarify`/`off_topic`, e os exemplos mostram o JSON exato (`search_term` sempre em inglês e preenchido nas intenções com ferramenta).
- Com o v2 no router, "Quem foi Helio Gracie?" passou a ser respondido com evidência da Wikipedia (bug do passo 3 resolvido).
- "Quem criou o berimbolo?" → classificação certa (`history`, `search_term='berimbolo'`), mas o resumo da Wikipedia não traz o criador e o Mestre respondeu "não tenho certeza" em vez de inventar. Limitação da fonte, não do classificador.
- CoT: o campo `reasoning` vem antes de `intent`. É curto de propósito: o gpt-oss já raciocina internamente; aqui o objetivo é auditar a decisão (ver acima por que cada caso foi classificado assim).


## 5. Parser com tratamento de erro (2.6)


In [14]:
from langchain_classic.output_parsers import OutputFixingParser, RetryWithErrorOutputParser
from langchain_core.exceptions import OutputParserException

# erros reais de LLM: valor fora do Literal + texto depois do JSON
bad_output = '{"level": "INTERMEDIATE", "context": "Escape from the mount"} Hope it helps!'

try:
    parser.parse(bad_output)
except OutputParserException as e:
    print("parser puro falhou:", str(e).splitlines()[0][:150])

fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm, max_retries=2)
fixed = fixing_parser.parse(bad_output)
print("\nOutputFixingParser:", fixed)


parser puro falhou: Failed to parse MestreResponse from completion {"level": "INTERMEDIATE", "context": "Escape from the mount"}. Got: 1 validation error for MestreRespon

OutputFixingParser: level='BEGINNER' context='I’m sorry, but I can only assist with martial arts technique questions.' steps=None common_mistake=None details=None variations=None safety=None clarifying_question=None


In [15]:
time.sleep(5)
question = {"question": "Sou faixa branca. Como escapo da montada?"}
prompt_value = chat_prompt.invoke(question)

truncated = '{"level": "BEGINNER", "context": "Você está por baixo da montada"'  # JSON cortado no meio

retry_parser = RetryWithErrorOutputParser.from_llm(parser=parser, llm=llm, max_retries=2)
retried = retry_parser.parse_with_prompt(truncated, prompt_value)
print("RetryWithErrorOutputParser:", retried.model_dump_json(indent=2, exclude_none=True))


RetryWithErrorOutputParser: {
  "level": "BEGINNER",
  "context": "Você está por baixo da montada"
}


In [16]:
def safe_answer(_: dict) -> MestreResponse:
    return MestreResponse(
        level="BEGINNER",
        context="I'm not sure about this detail right now. Please ask your academy instructor or try again later.",
    )

def build_safe_chain(model):
    return (chat_prompt | model | fixing_parser).with_retry(stop_after_attempt=2).with_fallbacks(
        [RunnableLambda(safe_answer)]
    )

safe_chain = build_safe_chain(llm)

broken_llm = ChatGroq(model="modelo-que-nao-existe", temperature=0, max_retries=0)  # simula API/modelo fora
broken_chain = build_safe_chain(broken_llm)

time.sleep(5)
print("normal :", safe_chain.invoke(question).context[:120])
print("quebrado:", broken_chain.invoke(question).context)


normal : Escapar da posição de montada, onde o adversário está sentado sobre o seu tronco, para recuperar a guarda ou ficar de jo
quebrado: I'm not sure about this detail right now. Please ask your academy instructor or try again later.


**Resultados:**
- Parser puro: `OutputParserException` (valor fora do `Literal` + texto depois do JSON).
- `OutputFixingParser`: corrige o formato, mas o conteúdo não é confiável, porque ele não vê a pergunta original. Em 2 execuções: na 1ª **inventou** `steps`/`common_mistake`/`safety` em inglês, tecnicamente fracos; na 2ª trocou o contexto por uma recusa ("I can only assist with martial arts technique questions"). Viola a regra anti-alucinação 1.
- `RetryWithErrorOutputParser`: com o prompt original, fechou o JSON truncado mantendo só o conteúdo existente (em PT). Igual nas 2 execuções. Conservador, sem invenção.
- `with_retry` + `with_fallbacks`: com um modelo inexistente, a chain devolveu um `MestreResponse` válido e seguro ("I'm not sure...") em vez de lançar exceção.

**Decisão:** retry > fixing neste projeto; fixing só para erro puramente de formato. A validação Pydantic é a última barreira, e o fallback garante que a interface nunca quebra. Custo: cada correção = 1 chamada extra de LLM.


## 6. Memória: buffer, summary e token (2.6)


In [17]:
import tiktoken
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory, ConversationTokenBufferMemory

# ChatGroq não sabe contar tokens sozinho -> contador com tiktoken (aproximação do tokenizer do gpt-oss)
enc = tiktoken.get_encoding("o200k_base")
llm_counter = ChatGroq(model=os.getenv("GROQ_MODEL"), temperature=0, custom_get_token_ids=enc.encode)

turns = [
    ("Sou faixa branca, 70kg, e treino com kimono.",
     "Ótimo! Com kimono você tem muitas pegadas para controlar o oponente. Em que posição você tem mais dificuldade?"),
    ("Estou na guarda fechada e o cara prendeu meu braço no chão.",
     "Se ele prendeu seu braço, cuidado com a passagem. Mantenha o cotovelo colado e use a outra mão para quebrar a postura dele."),
    ("Consegui quebrar a postura. E agora?",
     "Com a postura quebrada, você pode atacar o armlock ou o triângulo. Para o armlock, gire o quadril e passe a perna por cima da cabeça."),
    ("Prefiro algo mais seguro pro meu nível.",
     "Então a raspagem de tesoura é ideal: pegue gola e manga, coloque a canela na barriga dele e corte a perna de trás."),
]

memories = {
    "buffer": ConversationBufferMemory(),
    "summary": ConversationSummaryMemory(llm=llm),
    "token": ConversationTokenBufferMemory(llm=llm_counter, max_token_limit=120),
}

for name, mem in memories.items():
    for human, ai in turns:
        mem.save_context({"input": human}, {"output": ai})
        if name == "summary":
            time.sleep(3)  # summary chama o LLM a cada turno
    history = mem.load_memory_variables({})["history"]
    print(f"===== {name}: {llm_counter.get_num_tokens(history)} tokens\n{history}\n")


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_11524\2350528540.py:21: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  "summary": ConversationSummaryMemory(llm=llm),
C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_11524\2350528540.py:22: LangChainDeprecationWarning: The class `ConversationTokenBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and ht

===== buffer: 195 tokens
Human: Sou faixa branca, 70kg, e treino com kimono.
AI: Ótimo! Com kimono você tem muitas pegadas para controlar o oponente. Em que posição você tem mais dificuldade?
Human: Estou na guarda fechada e o cara prendeu meu braço no chão.
AI: Se ele prendeu seu braço, cuidado com a passagem. Mantenha o cotovelo colado e use a outra mão para quebrar a postura dele.
Human: Consegui quebrar a postura. E agora?
AI: Com a postura quebrada, você pode atacar o armlock ou o triângulo. Para o armlock, gire o quadril e passe a perna por cima da cabeça.
Human: Prefiro algo mais seguro pro meu nível.
AI: Então a raspagem de tesoura é ideal: pegue gola e manga, coloque a canela na barriga dele e corte a perna de trás.

===== summary: 196 tokens
The human says they are a white belt, weigh 70 kg, and train with a kimono. The AI notes that training with a kimono provides many grips for controlling the opponent and asks which position the human finds most difficult. The human report

In [18]:
mestre_memory_prompt = ChatPromptTemplate.from_messages([
    chat_prompt.messages[0],                     # mesmo system PTCF do Mestre
    ("system", "Conversation so far:\n{history}"),
    ("human", "{question}"),
]).partial(format_instructions=parser.get_format_instructions())

follow_up = {"question": "E se ele defender a raspagem, o que eu faço?"}

for name in ["buffer", "token"]:
    history = memories[name].load_memory_variables({})["history"]
    answer = (mestre_memory_prompt | llm | parser).invoke({**follow_up, "history": history})
    print(f"{name:7} level={answer.level} | {answer.context[:160]}")
    time.sleep(8)


buffer  level=BEGINNER | Você tentou a raspagem de tesoura da guarda fechada e o oponente bloqueou; aqui está o que fazer em seguida.
token   level=BEGINNER | Ele bloqueou sua raspagem de tesoura; você precisa recuperar a guarda e buscar outra oportunidade segura.


**Resultados (mesma conversa de 4 turnos):**

| Memória | Tokens no prompt | O que guardou | Custo extra |
| --- | --- | --- | --- |
| Buffer | 195 | tudo, literal | 0 |
| Summary | 191-196 (2 execuções) | tudo, resumido (em inglês) | 1 chamada de LLM por turno (4) |
| Token (limite 120) | 99 | só os 2 últimos turnos: esqueceu faixa branca, 70kg, kimono | 0 |

- No follow-up "E se ele defender a raspagem?", buffer e token citaram a raspagem de tesoura: a memória resolveu a referência.
- Summary ≈ buffer numa conversa curta: o resumo só compensa em conversa longa, onde o buffer cresce linear e estoura os 8000 tokens/min do Groq.
- O token é barato e previsível, mas perde o perfil do aluno. O `level=BEGINNER` saiu certo só pela regra "na dúvida, BEGINNER".

**Decisão para a fase 3:** janela curta (token/trim) para a conversa + perfil do aluno (faixa, peso, gi/no-gi, lesões) em memória de longo prazo separada (`docs/01_arquitetura.md` §6).

## 7. Ferramenta FAISS com índice em disco (2.6)

Fichas `data/kb/*.md` → pedaços por seção (`MarkdownHeaderTextSplitter`) → embeddings multilíngues (perguntas em PT, fichas em EN) → FAISS salvo em `data/index/techniques/`. A tool `search_technique` (em `src/tools.py`) carrega do disco, e a fase 4 reaproveita o índice sem reindexar.


In [19]:
import importlib
import src.tools
importlib.reload(src.tools)  # o kernel já importou a versão antiga de src.tools
from src.tools import INDEX_DIR, get_embeddings, load_technique_index, search_technique

from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.vectorstores import FAISS
from pathlib import Path

kb_files = sorted(Path("../data/kb").glob("*.md"))
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "technique"), ("##", "section")])

raw_docs, docs = [], []
for path in kb_files:
    for doc in splitter.split_text(path.read_text(encoding="utf-8")):
        doc.metadata["source"] = path.name
        raw_docs.append(doc.model_copy())
        # o splitter remove os títulos: sem isso, o pedaço "Common mistakes" não diz de qual técnica é
        doc.page_content = f"{doc.metadata['technique']} - {doc.metadata['section']}\n{doc.page_content}"
        docs.append(doc)

print(f"{len(docs)} chunks de {len(kb_files)} fichas")
print(docs[0].metadata, "|", docs[0].page_content[:80])

start = time.perf_counter()
FAISS.from_documents(docs, get_embeddings()).save_local(str(INDEX_DIR))
print(f"indexado em {time.perf_counter() - start:.1f}s ->", sorted(p.name for p in INDEX_DIR.iterdir()))


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_18136\3417708646.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


36 chunks de 6 fichas
{'technique': 'Armlock from Closed Guard', 'section': 'Position', 'source': 'armlock_closed_guard.md'} | Armlock from Closed Guard - Position
Bottom player in closed guard, legs locked 


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7286.42it/s]


indexado em 9.0s -> ['index.faiss', 'index.pkl']


In [ ]:
# por que o título entra no pedaço: mesma busca, índice sem e com o contexto do título
expected = {
    "como escapar da montada": "mount_escape_upa.md",
    "armlock from closed guard": "armlock_closed_guard.md",
    "o cara me pegou num triângulo, como defendo?": "triangle_closed_guard.md",
    "como faço a raspagem de tesoura": "scissor_sweep.md",
    "passagem de guarda de la riva": "dlr_guard_pass.md",
    "kimura": "kimura.md",
}

for name, chunks in [("sem título", raw_docs), ("com título", docs)]:
    vs = FAISS.from_documents(chunks, get_embeddings())
    hits = 0
    for q, source in expected.items():
        top = vs.similarity_search(q, k=1)[0].metadata["source"]
        hits += top == source
        print(f"  {name:10} | {q[:40]:40} -> {top}")
    print(f"{name}: ficha certa no topo {hits}/{len(expected)}\n")


  sem título | como escapar da montada                  -> dlr_guard_pass.md
  sem título | armlock from closed guard                -> armlock_closed_guard.md
  sem título | o cara me pegou num triângulo, como defe -> dlr_guard_pass.md
  sem título | como faço a raspagem de tesoura          -> dlr_guard_pass.md
  sem título | passagem de guarda de la riva            -> dlr_guard_pass.md
  sem título | kimura                                   -> dlr_guard_pass.md
sem título: ficha certa no topo 2/6

  com título | como escapar da montada                  -> mount_escape_upa.md
  com título | armlock from closed guard                -> armlock_closed_guard.md
  com título | o cara me pegou num triângulo, como defe -> triangle_closed_guard.md
  com título | como faço a raspagem de tesoura          -> scissor_sweep.md
  com título | passagem de guarda de la riva            -> dlr_guard_pass.md
  com título | kimura                                   -> kimura.md
com título: ficha certa no 

In [20]:
load_technique_index.cache_clear()  # força leitura do disco, não do objeto em memória

for q in ["como escapar da montada", "armlock from closed guard", "o cara me pegou num triângulo, como defendo?"]:
    print(f">>> {q}\n{search_technique.invoke(q)[:350]}\n" + "-" * 60)


>>> como escapar da montada
[mount_escape_upa.md / Steps]
Mount Escape: Upa (Bridge and Roll) - Steps
1. Protect first: elbows tight to the body, hands defending the neck, do not push the opponent's chest with straight arms.
2. Trap one arm: grab the opponent's wrist or overhook the arm (same side you will roll to) and pin it to your chest.
3. Trap the foot on the same side b
------------------------------------------------------------
>>> armlock from closed guard
[armlock_closed_guard.md / Safety]
Armlock from Closed Guard - Safety
Elbow locks can injure without warning. Apply slowly, release at the first tap, and tap early when caught. Train under supervision.

[armlock_closed_guard.md / Steps]
Armlock from Closed Guard - Steps
1. Control one arm: grab the opponent's sleeve or wrist and pull it across your
------------------------------------------------------------
>>> o cara me pegou num triângulo, como defendo?
[triangle_closed_guard.md / Common mistakes]
Triangle Choke from Clos

In [21]:
from langchain.agents import create_agent

MESTRE_RAG_PROMPT = """You are Mestre, a patient and direct Brazilian Jiu-Jitsu black belt instructor.

Tools:
- search_technique: ALWAYS use it to teach how to execute, defend or escape a technique. Base the steps on what it returns.
- search_video: ONLY when the student asks to see a video.
- search_wikipedia: history, athletes, academies.

Rules:
- Never invent steps that are not in the search_technique result; if it has nothing, say you are not sure.
- Never write a URL yourself.
- Always reply in the student's language."""

agent_rag = create_agent(llm, [search_video, search_wikipedia, search_technique], system_prompt=MESTRE_RAG_PROMPT)

time.sleep(5)
result = agent_rag.invoke({"messages": [("user", "Sou faixa branca. Como defendo o triângulo da guarda fechada?")]})
for m in result["messages"]:
    if getattr(m, "tool_calls", None):
        print("tool_call:", [(c["name"], c["args"]) for c in m.tool_calls])
print("\nresposta:", result["messages"][-1].content[:600])


tool_call: [('search_technique', {'query': 'triangle choke closed guard defense'})]

resposta: **Defesa do triângulo na guarda fechada (faixa branca)**  

Com base nas informações que encontrei, a defesa começa justamente evitando os erros mais comuns que o atacante costuma cometer ao aplicar o triângulo. Se você conseguir impedir que ele execute esses pontos fracos, a chance de escapar aumenta bastante.

| Erro comum do atacante | Como isso ajuda na sua defesa |
|------------------------|--------------------------------|
| **Travar a figura‑quatro (figure‑four) enquanto você está de frente para ele** – o choke fica fraco sem o ângulo correto. | **Mantenha o ângulo**: ao sentir o triâng


**Resultados:**
- 1ª versão do índice (pedaço = só o texto da seção): a ficha certa só vinha no topo em 2 de 6 perguntas. "como escapar da montada" trazia `dlr_guard_pass / Common mistakes`, porque o splitter remove os títulos e um pedaço "Common mistakes" não diz de qual técnica é. O agente chegou a chamar `search_technique` 4 vezes seguidas procurando a ficha certa e estourou o limite diário de tokens do Groq (ver passo 8).
- Correção: prefixar `técnica - seção` no texto de cada pedaço antes do embedding → ficha certa no topo em 6/6 (célula de comparação acima).
- O índice é recarregado do disco (`cache_clear` + `FAISS.load_local`) e as buscas em PT e EN trazem a ficha certa.
- Pendente: a célula do agente acima ainda mostra a execução com o índice antigo. Rodar de novo quando o limite diário do Groq liberar.

`allow_dangerous_deserialization=True`: o `index.pkl` é pickle. Só é seguro porque o índice é gerado pelo próprio projeto; nunca carregar índice de fonte desconhecida.

## 8. Callbacks customizados (2.6)

`MestreMonitor` registra, em tempo real, cada chamada de LLM (latência e tokens), cada tool chamada e cada erro, e avisa quando o consumo se aproxima do limite de 8000 tokens/min do Groq. Complementa o LangSmith: o callback é uma regra local que roda junto com a chain; o LangSmith fica para trace e avaliação (fase 5).


In [22]:
from langchain_core.callbacks import BaseCallbackHandler


class MestreMonitor(BaseCallbackHandler):
    """Latência e tokens por chamada de LLM, tools chamadas e erros; alerta perto do limite do Groq."""

    def __init__(self, token_budget: int = 8000):
        self.token_budget = token_budget
        self.llm_calls, self.tokens = 0, 0
        self.tools, self.errors = [], []
        self._start = {}

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self._start[run_id] = time.perf_counter()

    def on_llm_end(self, response, *, run_id, **kwargs):
        self.llm_calls += 1
        usage = getattr(response.generations[0][0].message, "usage_metadata", None) or {}
        self.tokens += usage.get("total_tokens", 0)
        elapsed = time.perf_counter() - self._start.pop(run_id, time.perf_counter())
        print(f"[LLM]   {elapsed:.1f}s | {usage.get('total_tokens', '?')} tokens | acumulado {self.tokens}")
        if self.tokens > 0.8 * self.token_budget:
            print(f"[WARN]  {self.tokens} tokens: perto do limite de {self.token_budget}/min do Groq")

    def on_tool_start(self, serialized, input_str, **kwargs):
        name = serialized.get("name", "?")
        self.tools.append(name)
        print(f"[TOOL]  {name}({input_str[:60]})")

    def on_llm_error(self, error, **kwargs):
        self.errors.append(repr(error))
        print(f"[ERROR] LLM: {str(error)[:120]}")

    def on_tool_error(self, error, **kwargs):
        self.errors.append(repr(error))
        print(f"[ERROR] tool: {str(error)[:120]}")

    def summary(self) -> str:
        return f"{self.llm_calls} chamadas de LLM | {self.tokens} tokens | tools: {self.tools} | erros: {len(self.errors)}"


In [23]:
time.sleep(10)
monitor_router = MestreMonitor()
for q in ["Quem foi Rickson Gracie?", "Me mostra um vídeo de kimura"]:
    print(f">>> {q}")
    router_v2.invoke({"question": q}, config={"callbacks": [monitor_router]})
print("router_v2:", monitor_router.summary())

time.sleep(10)
monitor_agent = MestreMonitor()
print("\n>>> agente RAG")
agent_rag.invoke(
    {"messages": [("user", "Sou faixa branca. Como escapo da montada?")]},
    config={"callbacks": [monitor_agent]},
)
print("agent_rag:", monitor_agent.summary())

monitor_broken = MestreMonitor()
print("\n>>> chain com modelo inexistente (fallback)")
broken_chain.invoke(question, config={"callbacks": [monitor_broken]})
print("broken_chain:", monitor_broken.summary())


>>> Quem foi Rickson Gracie?
[LLM]   0.9s | 1101 tokens | acumulado 1101
[TOOL]  search_wikipedia(Rickson Gracie)
[LLM]   0.8s | 509 tokens | acumulado 1610
>>> Me mostra um vídeo de kimura
[LLM]   1.4s | 1254 tokens | acumulado 2864
[TOOL]  search_video(kimura bjj)
router_v2: 3 chamadas de LLM | 2864 tokens | tools: ['search_wikipedia', 'search_video'] | erros: 0

>>> agente RAG
[LLM]   0.6s | 526 tokens | acumulado 526
[TOOL]  search_technique({'query': 'mount escape'})
[LLM]   0.5s | 636 tokens | acumulado 1162
[TOOL]  search_technique({'query': 'mount_escape_upa'})
[LLM]   0.6s | 767 tokens | acumulado 1929
[TOOL]  search_technique({'query': 'mount escape upa steps'})
[LLM]   0.5s | 1012 tokens | acumulado 2941
[TOOL]  search_technique({'query': 'mount escape upa steps'})
[ERROR] LLM: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m50sr


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m50srqt7ek0rcha135yp4ntz` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199129, Requested 2023. Please try again in 8m17.664s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

**Resultados:**
- Router: 2 chamadas de LLM no ramo history (classificar + responder com `search_wikipedia`) e 1 no ramo video (classificar + `search_video`, sem LLM para responder). ~1.100-1.250 tokens por classificação.
- Agente RAG (com o índice antigo, antes da correção do passo 7): 4 chamadas seguidas de `search_technique` procurando a ficha certa, até o Groq devolver `429` de **limite diário** (200.000 tokens/dia). O `on_llm_error` registrou o erro antes da exceção, que é exatamente o caso de uso do callback: mostrar onde o orçamento de tokens está sendo gasto.
